In [1]:
import requests
import pandas as pd
import time
import numpy as np
import concurrent.futures

In [24]:
df = pd.read_csv("chelsea_latlong.csv")


In [26]:
if "latitude" not in df.columns or "longitude" not in df.columns:
    raise ValueError("The dataset must contain 'latitude' and 'longitude' columns.")

def get_census_tract(lat, lon):
    base_url = "https://geocoding.geo.census.gov/geocoder/geographies/coordinates"
    params = {
        "x": f"{lon:.6f}",
        "y": f"{lat:.6f}",
        "benchmark": "Public_AR_Current",
        "vintage": "Current_Current",
        "format": "json"
    }

    try:
        response = requests.get(base_url, params=params, timeout=10)
        if response.status_code != 200:
            return "API Error"

        data = response.json()
        tracts = data.get("result", {}).get("geographies", {}).get("Census Tracts", [])

        if tracts:
            return tracts[0].get("NAME", "No Tract Found")  
        return "No Tract Found"

    except requests.exceptions.RequestException:
        return "Request Error"

def process_row(row):
    time.sleep(0.25)  
    return get_census_tract(row["latitude"], row["longitude"])

num_workers = 18 
with concurrent.futures.ThreadPoolExecutor(max_workers=num_workers) as executor:
    results = list(tqdm(executor.map(process_row, [row for _, row in df.iterrows()]), 
                        total=len(df), desc="Fetching Census Tracts"))

df["census_tract"] = results

df.to_csv("chelsea_census_tracts.csv", index=False)
print("Done! Census Tract names saved in 'chelsea_census_tracts.csv'.")


Fetching Census Tracts: 100%|██████████| 38202/38202 [19:49<00:00, 32.13it/s]  


Done! Census Tract names saved in 'dataset_with_census_tract.csv'.
